In [10]:
import pandas as pd
import requests
import psutil
import time
from pathlib import Path
import json
from datetime import datetime

In [13]:
OLLAMA_URL = "http://localhost:11434/api/generate"

MODELS = [
    "phi4-mini",
    "qwen2.5:7b-instruct",
    "qwen3:8b",
    "deepseek-r1:8b"
]

print(MODELS)

['phi4-mini', 'qwen2.5:7b-instruct', 'qwen3:8b', 'deepseek-r1:8b']


In [14]:
benchmark_dir = Path("../benchmarks")

BENCHMARKS = []

for file in sorted(benchmark_dir.glob("*.json")):

    print(f"Loading {file.name}")

    with open(file, "r", encoding="utf-8") as f:

        data = json.load(f)

        if isinstance(data, list):
            BENCHMARKS.extend(data)

        elif (
            isinstance(data, dict)
            and "benchmarks" in data
        ):
            BENCHMARKS.extend(data["benchmarks"])

print("="*70)
print(f"Loaded {len(BENCHMARKS)} benchmark prompts.")
print("="*70)

for bench in BENCHMARKS:
    print(
        bench["category"],
        "->",
        bench["name"]
    )

Loading extraction.json
Loading math.json
Loading rag.json
Loading reasoning.json
Loading summarization.json
Loaded 10 benchmark prompts.
extraction -> invoice_fields
extraction -> person_location
math -> average_speed
math -> probability
rag -> context_only
rag -> unknown_answer
reasoning -> river_crossing
reasoning -> calendar_logic
summarization -> bullet_summary
summarization -> one_sentence


In [ ]:
def run_model(model, prompt):

    start = time.time()

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 128
            }
        },
        timeout=180
    )

    latency = round(
        time.time() - start,
        2
    )

    r = response.json()

    eval_duration = r.get(
        "eval_duration",
        0
    )

    prompt_eval_duration = r.get(
        "prompt_eval_duration",
        0
    )

    total_duration = r.get(
        "total_duration",
        0
    )

    load_duration = r.get(
        "load_duration",
        0
    )

    return {

        "response":
            r.get("response",""),

        "thinking":
            r.get("thinking",""),

        "latency_sec":
            latency,

        "ttft_sec":
            round(
                prompt_eval_duration/1e9,
                3
            ),

        "total_duration_sec":
            round(
                total_duration/1e9,
                3
            ),

        "load_duration_sec":
            round(
                load_duration/1e9,
                3
            ),

        "prompt_tokens":
            r.get(
                "prompt_eval_count",
                0
            ),

        "output_tokens":
            r.get(
                "eval_count",
                0
            ),

        "tokens_per_sec":
            round(
                r.get(
                    "eval_count",
                    0
                ) /
                max(
                    eval_duration/1e9,
                    1e-9
                ),
                2
            ),

        "done_reason":
            r.get(
                "done_reason",
                ""
            ),

        "created_at":
            r.get(
                "created_at",
                ""
            )
    }

In [15]:
results = []

for model in MODELS:

    print("="*80)
    print(model)
    print("="*80)

    for bench in BENCHMARKS:

        print(
            f"Running {bench['category']} -> {bench['name']}"
        )

        output = run_model(
            model,
            bench["prompt"]
        )

        results.append({

            "model":
                model,

            "category":
                bench["category"],

            "benchmark":
                bench["name"],

            **output

        })

print("\nBenchmark Finished.")

phi4-mini
Running extraction -> invoice_fields
Running extraction -> person_location
Running math -> average_speed
Running math -> probability
Running rag -> context_only
Running rag -> unknown_answer
Running reasoning -> river_crossing
Running reasoning -> calendar_logic
Running summarization -> bullet_summary
Running summarization -> one_sentence
qwen2.5:7b-instruct
Running extraction -> invoice_fields
Running extraction -> person_location
Running math -> average_speed
Running math -> probability
Running rag -> context_only
Running rag -> unknown_answer
Running reasoning -> river_crossing
Running reasoning -> calendar_logic
Running summarization -> bullet_summary
Running summarization -> one_sentence
qwen3:8b
Running extraction -> invoice_fields
Running extraction -> person_location
Running math -> average_speed
Running math -> probability
Running rag -> context_only
Running rag -> unknown_answer
Running reasoning -> river_crossing
Running reasoning -> calendar_logic
Running summariz

In [16]:
df = pd.DataFrame(results)
df

,model,category,benchmark,response,thinking,latency_sec,ttft_sec,total_duration_sec,load_duration_sec,prompt_tokens,output_tokens,tokens_per_sec,done_reason,created_at
0,phi4-mini,extraction,invoice_fields,"```json\n\n{\n\n ""invoice_number"": ""12345"",...",,6.61,2.028,6.602,1.390,33,35,11.13,stop,2026-06-27T14:53:25.537909709Z
1,phi4-mini,extraction,person_location,"```json\n{\n ""person"": ""Alice"",\n ""origin"": ...",,4.42,1.522,4.418,0.181,26,30,11.19,stop,2026-06-27T14:53:29.959779088Z
2,phi4-mini,math,average_speed,80 km/h,,2.65,2.188,2.646,0.180,37,4,14.85,stop,2026-06-27T14:53:32.610082755Z
3,phi4-mini,math,probability,The total number of ways to draw two balls fro...,,19.80,2.026,19.796,0.187,35,184,10.58,stop,2026-06-27T14:53:52.409803798Z
4,phi4-mini,rag,context_only,Paris,,1.89,1.605,1.889,0.191,27,2,22.61,stop,2026-06-27T14:53:54.303350275Z
5,phi4-mini,rag,unknown_answer,Guido van Rossum.,,2.35,1.583,2.342,0.186,29,7,12.44,stop,2026-06-27T14:53:56.649301565Z
6,phi4-mini,reasoning,river_crossing,The solution to this classic puzzle is as foll...,,26.51,1.460,26.506,0.189,25,256,10.41,length,2026-06-27T14:54:23.159155065Z
7,phi4-mini,reasoning,calendar_logic,To find out which day of the week it would fal...,,26.77,1.418,26.762,0.186,24,256,10.29,length,2026-06-27T14:54:49.925811794Z
8,phi4-mini,summarization,bullet_summary,- AI allows computers to execute complex funct...,,5.92,1.706,5.917,0.188,29,44,11.08,stop,2026-06-27T14:54:55.847022078Z
9,phi4-mini,summarization,one_sentence,Large language models (LLMs) like GPT-3 and BE...,,7.74,1.342,7.740,0.192,23,67,10.93,stop,2026-06-27T14:55:03.591197114Z


In [19]:
results_dir = Path("../results")

results_dir.mkdir(
    exist_ok=True
)

csv_file = (
    results_dir
    /
    "benchmark_results.csv"
)

df.to_csv(
    csv_file,
    index=False
)

print(csv_file)

../results/benchmark_results.csv


In [20]:
leaderboard = (

    df.groupby("model")

      .agg(

          benchmarks=(
              "benchmark",
              "count"
          ),

          avg_latency=(
              "latency_sec",
              "mean"
          ),

          avg_ttft=(
              "ttft_sec",
              "mean"
          ),

          avg_tps=(
              "tokens_per_sec",
              "mean"
          ),

          avg_prompt_tokens=(
              "prompt_tokens",
              "mean"
          ),

          avg_output_tokens=(
              "output_tokens",
              "mean"
          )

      )

      .round(2)

      .sort_values(
          "avg_latency"
      )

)

leaderboard

,benchmarks,avg_latency,avg_ttft,avg_tps,avg_prompt_tokens,avg_output_tokens
model,,,,,,
phi4-mini,10,10.47,1.69,12.55,28.8,88.5
qwen2.5:7b-instruct,10,13.85,3.99,7.88,57.0,43.2
qwen3:8b,10,47.16,4.39,5.66,38.0,222.5
deepseek-r1:8b,10,48.45,3.58,5.80,30.0,240.5


In [21]:
leaderboard_file = (
    results_dir
    /
    "leaderboard.csv"
)

leaderboard.to_csv(
    leaderboard_file
)

print(leaderboard_file)

../results/leaderboard.csv


In [22]:
import platform
import psutil

print("="*60)

print(
    "Platform:",
    platform.platform()
)

print(
    "Processor:",
    platform.processor()
)

print(
    "Physical cores:",
    psutil.cpu_count(False)
)

print(
    "Logical cores:",
    psutil.cpu_count()
)

print(
    "RAM:",
    round(
        psutil.virtual_memory().total
        /1024**3,
        2
    ),
    "GB"
)

Platform: Linux-6.17.0-1018-azure-x86_64-with-glibc2.39
Processor: x86_64
Physical cores: 2
Logical cores: 4
RAM: 15.57 GB
